# Chapter 3 · learn, drive, cooperate
**Student workbook · five exercise blocks · NumPy + PyTorch**

Learn how agents make decisions in a dessert café, highway traffic, and two social games.
Run this workbook **section by section**. Fill each TODO, then run its check cell.
An unfinished TODO intentionally raises `NotImplementedError`; there is no answer switch.
You can use Run All after completing the exercises.

![Four tasks: café inventory decisions, highway navigation, repeated social incentives, and risky coordination.](../assets/figures/four_tasks_overview.png)

**Learning goals:** index and mask NumPy arrays; implement a tabular TD update;
read a PyTorch network's input/output shapes; build DQN targets and loss;
explain how another agent changes the decision problem.

Synthetic café demand and prices are teaching assumptions. See the repository README for local setup.

## 0 · Start the local kernel
Create the environment once using the repository README, then choose the **Python (RL Chapter 3)** kernel.
This notebook checks its environment; it does not install or upgrade packages.
Widgets run in JupyterLab. Ordinary code alternatives accompany the controls.

In [ ]:
import os, sys, copy, json, time
from pathlib import Path
from importlib.metadata import version

# Find the project from either the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'rl_lab').is_dir() and (p / 'environment.yml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open JupyterLab from the cloned repository; see README.md.')
sys.path.insert(0, str(ROOT))
import rl_lab  # Sets headless rendering before importing pygame or matplotlib.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display, Image, Markdown
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
from rl_lab.rules import LearningRules
from rl_lab.checks import (check_foundations, check_dqn,
                           check_tit_for_tat, check_expected_payoffs)
from rl_lab.cafe import (DessertCafe, CafeQAgent, cafe_episode, cafe_evaluate,
                         train_cafe, plot_cafe, plot_cafe_policy)
from rl_lab.highway import (HIGHWAY_CONFIG, ACTION_NAMES, make_highway, random_argmax,
                            highway_comparison, plot_highway_comparison,
                            highway_rollout, highway_evaluate, PlaybackDQN,
                            load_cached_comparison)
from rl_lab.social import (read_payoffs, plot_payoffs, social_match,
                           train_social, plot_social, plot_stag_threshold)
from rl_lab.practice import cafe_controls, dilemma_controls, stag_controls

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(f'Select the Python 3.11 lab kernel. Current Python: {sys.executable}')
torch.set_num_threads(1)
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False,
                     'axes.spines.right': False})
VERSIONS = {name: version(name) for name in
            ['numpy','torch','gymnasium','highway-env','pettingzoo','matplotlib','pandas']}
print('Kernel:', sys.executable)
print('Versions:', VERSIONS)

In [ ]:
# 'smoke': fast code check; 'teaching': video/practice; 'experiment': longer multi-seed run.
RUN_MODE = os.environ.get('CH3_RUN_MODE', 'teaching')
RUN_HIGHWAY_TRAINING = os.environ.get('CH3_TRAIN_HIGHWAY', '0') == '1'
RUN_SOCIAL_TRAINING = False   # Optional extension, after the core lesson.
SHOW_WIDGETS = True
DEVICE = 'cpu'
if RUN_MODE not in {'smoke','teaching','experiment'}:
    raise ValueError('Choose smoke, teaching, or experiment.')
BUDGETS = {
    'smoke': dict(cafe=200, highway=256, eval_n=2, seeds=[0], social=20),
    'teaching': dict(cafe=20000, highway=2000, eval_n=10, seeds=[0], social=300),
    'experiment': dict(cafe=60000, highway=20000, eval_n=30, seeds=[0,1,2], social=1500),
}
budget = BUDGETS[RUN_MODE]
OUTPUT_DIR = rl_lab.RESULTS / (RUN_MODE + '_' + time.strftime('%Y%m%d_%H%M%S'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Budget:', budget)
print('Highway comparison:', 'TRAIN NEW AGENTS' if RUN_HIGHWAY_TRAINING else 'LOAD RECORDED TEACHING RUN')
print('Results directory:', OUTPUT_DIR)

`smoke` checks code paths and is too small to judge learning. The default highway display
loads a recorded, matched-budget teaching run to keep the recording moving.
Set `RUN_HIGHWAY_TRAINING = True` to train with your own functions, and rerun from setup.
`experiment` increases the budget, but still needs the training flag to start new highway training.
No ranking should be inferred from a single training seed.

## 1 · The array-to-tensor bridge

**NumPy:** an array stores numbers; a mask selects allowed entries; indexing selects a row or action.
Here a row is a state and a column is an action. Predict the printed shapes before running.

In [ ]:
q_demo = np.array([[1., 4., 2.], [3., -1., 5.]], dtype=np.float32)
valid_actions = np.array([True, False, True])
print('Q-table shape:', q_demo.shape)
print('Values in state 0:', q_demo[0])
print('Allowed action indices:', np.flatnonzero(valid_actions))
print('Allowed values in state 0:', q_demo[0, valid_actions])
print('Masking does not change the original table:', q_demo)
# Try: change the mask, then predict which action is feasible AND most valuable.

**PyTorch:** a tensor also has a shape and dtype. With `requires_grad=True`, PyTorch tracks
operations so `backward()` can calculate gradients. An optimizer uses those gradients to
change parameters. A forward pass alone does not train a network.

In [ ]:
torch.manual_seed(0)
x = torch.tensor([[1., 2.], [3., 4.]], dtype=torch.float32)  # batch of 2 examples
tiny_model = nn.Linear(2, 1)
optimizer_demo = torch.optim.SGD(tiny_model.parameters(), lr=.01)
prediction = tiny_model(x)
target_demo = torch.tensor([[1.], [2.]])
loss_demo = ((prediction - target_demo) ** 2).mean()
optimizer_demo.zero_grad()
loss_demo.backward()
print('Input:', tuple(x.shape), 'prediction:', tuple(prediction.shape))
print('Weight gradient:', tiny_model.weight.grad)
optimizer_demo.step()
print('One parameter update completed.')
# Predict: what would happen if optimizer_demo.step() were omitted?

**Keep this reference nearby:**

| Operation | NumPy | PyTorch |
|---|---|---|
| Create floating values | `np.array(data, dtype=np.float32)` | `torch.tensor(data, dtype=torch.float32)` |
| Inspect dimensions | `x.shape` | `x.shape` |
| Maximum per row | `x.max(axis=1)` | `x.max(dim=1).values` |
| Integer action indices | `np.int64` | `torch.long` |
| Matrix multiplication | `a @ b` | `a @ b` |
| Stop gradient tracking | No automatic gradients | `with torch.no_grad():` or `x.detach()` |

Neither `axis=1` nor `dim=1` means “time” automatically: it means the second array dimension.
For a batch of action values, that dimension represents actions.

## 2 · Café Q-learning

Manage Dubai chewy cookies (두쫀쿠) and pistachio cake. Cake demand peaks earlier; cookie demand peaks later.
The café is a small, finite-horizon decision problem.

| Element | Meaning |
|---|---|
| State | Hour, cookie inventory, cake inventory, remaining ingredients |
| Action | Prepare `(0,0), (2,0), (0,2), (4,0), (2,2), (0,4)` portions |
| Timing | Sell current stock this hour; new preparation arrives next hour |
| Resources | 24 preparation units during the day; capacity 8 per dessert |
| Opening | 2 of each dessert, prepared before opening and charged at cost |
| Prices / costs | Cookie: 6 / 3 credits; cake: 7 / 4 credits |
| Reward | Cash profit minus 0.5 credits per unserved customer |
| Closing | 6 p.m.; discard stock for 1 credit per portion; no final-hour preparation |

Your action must respect the ingredient, capacity, and final-hour mask.
The simulator pre-generates customers so policies can be evaluated on the same customer days.

### Exercise 1 · exploration and a Q-learning update

Implement three functions. Exploration must also respect the action mask.
Break equal-value greedy ties randomly; otherwise the first action gets an accidental advantage.


\[
y=r+\gamma(1-\mathrm{terminated})\max_{a'\,\mathrm{valid}}Q(s',a'),
\qquad Q(s,a)\leftarrow Q(s,a)+\alpha[y-Q(s,a)].
\]

**Hint:** `np.flatnonzero(mask)` gives feasible action indices. The largest value may be negative.

In [ ]:
def epsilon_greedy(values, epsilon, rng, mask=None):
    # TODO: explore with probability epsilon; otherwise choose a feasible maximum.
    # Use random tie-breaking. If mask is None, all actions are feasible.
    raise NotImplementedError('Exercise 1: implement epsilon_greedy, then run its check cell.')

def td_target(reward, terminated, next_values, next_mask, gamma):
    # TODO: return immediate reward at true termination; otherwise bootstrap over valid next actions.
    raise NotImplementedError('Exercise 1: implement td_target.')

def q_update(old_value, target, alpha):
    # TODO: move old_value an alpha fraction toward target.
    raise NotImplementedError('Exercise 1: implement q_update.')

In [ ]:
check_foundations(epsilon_greedy, td_target, q_update)
rules = LearningRules(epsilon_greedy, td_target, q_update)

### Exercise 2 · run the café, then explain the policy

Before viewing demand, choose how many desserts to prepare.
The controls reveal each hour's outcome only after your decision.
Try the same seed twice and explain one change in your plan.

In [ ]:
if SHOW_WIDGETS:
    cafe_ui = cafe_controls(seed=2026)

**Ordinary-code alternative:** edit the eight action indices below and rerun.
An infeasible choice becomes waiting in this manual demo, and is explicitly reported.
The simulator itself rejects infeasible actions.

In [ ]:
MANUAL_ACTIONS = [2, 5, 4, 1, 1, 3, 0, 0]
manual_env = DessertCafe()
invalid_requests = []
def manual_policy(state, mask):
    requested = MANUAL_ACTIONS[int(state[0])]
    if requested not in range(len(mask)) or not mask[requested]:
        invalid_requests.append((10 + int(state[0]), requested))
        return 0
    return int(requested)
manual_return, manual_rows = cafe_episode(manual_env, manual_policy, seed=2026)
display(pd.DataFrame([{
    'hour': r['hour'], 'stock': tuple(r['inventory']), 'prepare_next': tuple(r['produced']),
    'demand': tuple(r['demand']), 'sales': tuple(r['sales']), 'cash_profit': r['cash_profit']
} for r in manual_rows]))
print('Return:', manual_return, '| Requests replaced by waiting:', invalid_requests)
manual_env.close()

The provided agent stores `Q[hour, cookies, cakes, ingredients, action]` in a NumPy array.
The training loop below shows where your three functions enter. The full simulator and
bookkeeping are in `rl_lab/cafe.py`.

In [ ]:
# One learning transition, with all three student functions visible.
demo_env = DessertCafe()
demo_agent = CafeQAgent(demo_env, rules, seed=0)
state, info = demo_env.reset(seed=0)
action = epsilon_greedy(demo_agent.q[tuple(state)], .2, demo_agent.rng, info['action_mask'])
next_state, reward, terminated, truncated, next_info = demo_env.step(action)
target = td_target(reward, terminated, demo_agent.q[tuple(next_state)], next_info['action_mask'], demo_agent.gamma)
index = (*state, action)
demo_agent.q[index] = q_update(demo_agent.q[index], target, demo_agent.alpha)
print('Updated index:', index, '| target:', target, '| Q:', demo_agent.q[index])
demo_env.close()

In [ ]:
cafe_agent, cafe_history = train_cafe(rules, episodes=budget['cafe'], seed=0)
cafe_eval = {}
for label, policy in [('Tabular Q-learning', cafe_agent), ('Forecast-based rule', None)]:
    cafe_eval[label] = cafe_evaluate(policy, range(20000, 20000 + (20 if RUN_MODE == 'smoke' else 100)))
cafe_scoreboard = pd.DataFrame({name: frame.mean() for name, frame in cafe_eval.items()}).T
display(cafe_scoreboard.rename(columns={'return_':'return'}).round(2))
cafe_example = plot_cafe(cafe_agent, cafe_history, save_dir=OUTPUT_DIR)
plot_cafe_policy(cafe_agent, ingredients=12, other_stock=2, save_dir=OUTPUT_DIR)

**Write your explanation:** Does the learned policy anticipate each demand peak?
Compare reward, cash profit, missed customers, and waste. Why might the forecast rule win?
What does a gray policy-map cell mean?

**Optional experiment:** increase `missed_penalty` to 2.0, retrain, and compare profit and missed customers.
Do not directly compare raw rewards from two different reward definitions.

In [ ]:
RUN_CAFE_REWARD_CHALLENGE = False
if RUN_CAFE_REWARD_CHALLENGE:
    changed_agent, _ = train_cafe(rules, episodes=budget['cafe'], seed=0,
                                 env_kwargs={'missed_penalty': 2.0})
    changed_scores = cafe_evaluate(changed_agent, range(20000,20100),
                                   env_kwargs={'missed_penalty':2.0})
    display(pd.DataFrame({'original':cafe_eval['Tabular Q-learning'].mean(),
                          'retrained_service_priority':changed_scores.mean()}).T[['profit','missed','waste']])

## 3 · Highway navigation with a DQN

The existing `highway-fast-v0` task has five actions: lane left, maintain, lane right, faster, slower.
The return rewards progress/speed and penalizes crashes. A high return alone does not establish safety.

Each observation has **5 vehicles × 5 features = 25 numbers**. Presence identifies real rows;
the remaining features describe position and velocity. Both methods receive the same features.
Tabular Q-learning bins them; DQN approximates values with a neural network.

**Predict:** What happens to table coverage when you make every bin finer?
Can a network return a value for an observation it has never seen exactly before?

In [ ]:
inspection_env = make_highway(render=True)
sample_observation, _ = inspection_env.reset(seed=0)
display(pd.DataFrame(sample_observation.reshape(5,5),
    columns=['presence','x / 100m','y / lane width','vx / 30m/s','vy / 10m/s']))
fig, ax = plt.subplots(figsize=(10,3))
ax.imshow(inspection_env.render()); ax.axis('off'); ax.set_title('HighwayEnv · seeded initial scene')
fig.savefig(OUTPUT_DIR / 'highway_scene.png', dpi=180)
plt.show()
print('Input shape:', sample_observation.shape, '| Actions:', ACTION_NAMES)
inspection_env.close()

### From a table row to network outputs
`QNetwork` maps 25 features to five action values. A batch has shape `(batch_size, 25)`;
the output has shape `(batch_size, 5)`. Actions are integer column indices.

The provided replay buffer stores transitions and samples batches. Read its five returned tensors:
states, actions, rewards, next states, and **true termination** flags.

In [ ]:
class QNetwork(nn.Module):
    def __init__(self, obs_dim=25, n_actions=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, n_actions),
        )

    def forward(self, x):
        # (batch, 25) -> (batch, 5); no softmax: these are values, not probabilities.
        return self.net(x)


class ReplayBuffer:
    """Provided storage: keep a bounded history and sample training batches."""
    def __init__(self, capacity=20000, obs_dim=25, seed=0):
        self.states = np.zeros((capacity, obs_dim), dtype=np.float32)
        self.next_states = np.zeros_like(self.states)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.terminated = np.zeros(capacity, dtype=np.float32)
        self.capacity = capacity
        self.position = 0
        self.size = 0
        self.rng = np.random.default_rng(seed)

    def add(self, state, action, reward, next_state, terminated):
        i = self.position
        self.states[i] = state
        self.actions[i] = action
        self.rewards[i] = reward
        self.next_states[i] = next_state
        self.terminated[i] = terminated  # Do not store terminated OR truncated here.
        self.position = (i + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size, device):
        indices = self.rng.integers(self.size, size=batch_size)
        arrays = (self.states, self.actions, self.rewards,
                  self.next_states, self.terminated)
        return tuple(torch.as_tensor(array[indices], device=device) for array in arrays)

In [ ]:
torch.manual_seed(0)
shape_demo = QNetwork()
batch_demo = torch.zeros((4,25), dtype=torch.float32)
print('Network:', shape_demo)
print('Batch output shape:', tuple(shape_demo(batch_demo).shape))

### Exercise 3 · DQN targets and loss

Complete the target and the loss. `next_values` already contains the **target network's**
maximum next-action value. Return one target per transition, then a scalar mean Huber loss.

\[
y_i=r_i+\gamma(1-\mathrm{terminated}_i)\max_{a'}Q_{\theta^-}(s'_i,a').
\]

Stop an episode after termination **or** truncation. Remove the bootstrap term only for
true termination: the highway time limit is an external cutoff.
The finite café horizon is true termination, with time included in its state.

**Hints:** `q_values.gather(1, actions.long().reshape(-1,1)).squeeze(1)` selects one output
per example. `nn.functional.smooth_l1_loss` is Huber loss. Targets must not receive gradients.

In [ ]:
def dqn_targets(rewards, terminated, next_values, gamma):
    # TODO: produce a vector of targets, masking only TRUE termination.
    raise NotImplementedError('Exercise 3: implement dqn_targets.')

def dqn_loss(q_values, actions, targets):
    # TODO: select each chosen Q(s,a); compare with detached targets using mean Huber loss.
    raise NotImplementedError('Exercise 3: implement dqn_loss.')

In [ ]:
check_dqn(dqn_targets, dqn_loss)

### Read one training update
The online network receives gradients; the target network is periodically copied from it.
The replay buffer allows old transitions to be used again. Locate `no_grad`, `backward`,
`optimizer.step`, and `sync_target` in the provided scaffold.

In [ ]:
class DQNAgent:
    def __init__(self, seed=0, gamma=.95, learning_rate=1e-3, device='cpu'):
        torch.manual_seed(seed)
        self.device = torch.device(device)
        self.online = QNetwork().to(self.device)
        self.target = copy.deepcopy(self.online).to(self.device)
        self.target.eval()
        for parameter in self.target.parameters():
            parameter.requires_grad_(False)
        self.optimizer = torch.optim.Adam(self.online.parameters(), lr=learning_rate)
        self.buffer = ReplayBuffer(seed=seed + 17)
        self.rng = np.random.default_rng(seed)
        self.gamma = gamma
        self.last_loss = np.nan

    def act(self, state, epsilon=0.):
        if self.rng.random() < epsilon:
            return int(self.rng.integers(5))
        with torch.no_grad():
            state_tensor = torch.as_tensor(state, dtype=torch.float32, device=self.device)
            values = self.online(state_tensor).cpu().numpy()
        return random_argmax(values, self.rng)

    def learn(self, batch_size=64):
        states, actions, rewards, next_states, terminated = self.buffer.sample(batch_size, self.device)
        # Fixed targets: gradients must not flow through the target network.
        with torch.no_grad():
            next_values = self.target(next_states).max(dim=1).values
            targets = dqn_targets(rewards, terminated, next_values, self.gamma)

        loss = dqn_loss(self.online(states), actions, targets)
        self.optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), 10.)
        self.optimizer.step()
        self.last_loss = float(loss.detach().cpu())
        return self.last_loss

    def sync_target(self):
        self.target.load_state_dict(self.online.state_dict())

### Compare methods on the same interaction budget

The recorded teaching comparison trains coarse, medium, fine, and DQN agents for **2,000
interactions each, one training seed**. These are intentionally short runs and may perform poorly.
The figure reports evaluation return versus interactions and training time, plus crashes,
speed, and tabular evaluation-state coverage. Table coverage is not a metric for DQN.

To train your implementation, enable `RUN_HIGHWAY_TRAINING` in setup and rerun.
In experiment mode, the shaded region is the range over training seeds, not a confidence interval.
Evaluation uses separate seeds and does not update models or add table entries.

In [ ]:
highway_agents = {}
if RUN_HIGHWAY_TRAINING:
    highway_agents, highway_curves, highway_scores = highway_comparison(
        rules, DQNAgent, steps=budget['highway'], seeds=budget['seeds'],
        eval_n=budget['eval_n'], device=DEVICE)
    highway_source = 'trained in this run'
else:
    highway_curves, highway_scores = load_cached_comparison()
    highway_source = 'recorded teaching run: 2,000 interactions/method, seed 0'
print('Comparison source:', highway_source)
highway_summary = plot_highway_comparison(highway_curves, highway_scores, save_dir=OUTPUT_DIR)

In [ ]:
if highway_agents:
    print('DQN trained in THIS run:', budget['highway'], 'interactions')
    current_rollout = highway_rollout(highway_agents[('DQN', budget['seeds'][0])],
                                      seed=2026, save_path=OUTPUT_DIR / 'current_dqn.gif')
    print('Return:', current_rollout.reward.sum(), '| Crash:', current_rollout.crashed.any())
else:
    print('No new highway training requested. The next rollout is a separately labelled reference.')

### Inspect a separately trained reference policy

This supplied checkpoint used **20,000 interactions, seed 0** and the same configuration.
It is a playback example and is **not** part of the 2,000-interaction comparison above.
Its original metadata and a new local re-evaluation are stored beside the assets.
One seed or a successful animation does not establish reliable performance.

In [ ]:
reference_agent = PlaybackDQN()
reference_metadata = json.loads((ROOT / 'assets/checkpoints/dqn_reference_20k_seed0.json').read_text())
print('Reference training budget:', reference_metadata['steps'], '| training seed:', reference_metadata['seed'])
reference_rollout = highway_rollout(reference_agent, seed=2026,
                                    save_path=OUTPUT_DIR / 'reference_dqn.gif')
print('Shown episode return:', reference_rollout.reward.sum(), '| Crash:', reference_rollout.crashed.any())

**Explain:** Which binning gives the best coverage? Does higher coverage always mean higher return?
Why report collisions as well as reward? Why can't we include the 20,000-step checkpoint in
the short-run algorithm ranking? Write two sentences before discussing.

## 4 · Prisoner's Dilemma: incentives and repeated responses

Both players choose simultaneously from past information. Read each entry as **(Player A, Player B)**.
The game is general-sum: individual reward is neither a shared team score nor necessarily the
negative of the other player's reward. We use the unchanged bundled PettingZoo-Dilemmas game.

### Exercise 4 · tit-for-tat
Action `0` is cooperate; `1` is defect. Marker `2` means there was no previous partner action.
Start by cooperating; afterwards copy the partner's previous action.
Before running, predict which strategy wins against an always-cooperating partner.

In [ ]:
def tit_for_tat(previous_opponent_action):
    # TODO: cooperate on the opening round; then copy the partner's PREVIOUS action.
    raise NotImplementedError('Exercise 4: implement tit_for_tat.')

In [ ]:
check_tit_for_tat(tit_for_tat)
pd_matrix = plot_payoffs('prisoners_dilemma')
strategies = {'always cooperate':'always_cooperate',
              'always defect':'always_defect', 'tit-for-tat':tit_for_tat}
tournament = []
for name, strategy in strategies.items():
    for opponent_name, opponent in strategies.items():
        match = social_match('prisoners_dilemma',strategy,opponent,nrounds=20)
        tournament.append(dict(strategy=name,opponent=opponent_name,
                               mean_reward_A=match.reward_a.mean(),
                               mutual_cooperation=match.both_cooperate.mean()))
tournament = pd.DataFrame(tournament)
display(tournament.pivot(index='strategy',columns='opponent',values='mean_reward_A'))
if SHOW_WIDGETS:
    dilemma_ui = dilemma_controls(tit_for_tat)

**Explain:** Does tit-for-tat earn the most against every opponent? How does a present action
affect a reactive partner's later response? What is different from a one-shot game?
The tournament is a strategy comparison; it is not itself RL training.

In [ ]:
if RUN_SOCIAL_TRAINING:
    pd_agent_a, pd_agent_b, pd_history = train_social('prisoners_dilemma',rules,
        episodes=budget['social'],opponent='learner',seed=0)
    pd_match = social_match('prisoners_dilemma',pd_agent_a,pd_agent_b)
    plot_social(pd_history,pd_match,'prisoners_dilemma')
else:
    print('Optional extension: enable RUN_SOCIAL_TRAINING to train two independent Q-learners.')

## 5 · Stag Hunt: coordination under risk

Both players prefer the high stag payoff when the other also chooses stag. Hare can be a
better individual response when the partner is unlikely to coordinate.
The actual upstream payoffs are shown below.

### Exercise 5 · expected payoff
Let `p` be the probability the partner chooses stag. Return a NumPy array with Player A's
expected payoff for **each** action. Matrix axes are `[A action, B action, player]`.
Action `0` is stag and `1` is hare.

**Hint:** weight each row of Player A's payoff matrix by `[p, 1-p]`.
Predict approximately where the preferred action changes before moving the slider.

In [ ]:
def expected_payoffs(matrix, p_partner_stag):
    # TODO: return two expected PLAYER-A payoffs, weighted by the partner's choices.
    raise NotImplementedError('Exercise 5: implement expected_payoffs.')

In [ ]:
stag_matrix = plot_payoffs('stag_hunt')
check_expected_payoffs(expected_payoffs,stag_matrix)
stag_threshold = plot_stag_threshold(stag_matrix,expected_payoffs,
                                     save_path=OUTPUT_DIR / 'stag_threshold.png')
# Ordinary-code experiment: edit p and rerun.
p = .5
print('At p =', p, '| expected [stag, hare] =', expected_payoffs(stag_matrix,p))
if SHOW_WIDGETS:
    stag_ui = stag_controls(expected_payoffs)

**Explain:** Why can individually reasonable decisions lead to the lower joint payoff?
Name one intervention that might help players coordinate. Does a larger joint reward
automatically give the agents a shared reward signal?

In [ ]:
if RUN_SOCIAL_TRAINING:
    stag_results = []
    for preference, label in [(0,'Initially favor stag'),(1,'Initially favor hare')]:
        for seed in ([0,1,2] if RUN_MODE == 'experiment' else [0]):
            a,b,history = train_social('stag_hunt',rules,episodes=budget['social'],
                seed=seed,initial_preference=preference,epsilon_start=.2)
            match = social_match('stag_hunt',a,b,nrounds=20)
            stag_results.append(dict(initialization=label,seed=seed,
                mutual_stag=match.both_cooperate.mean(),reward_A=match.reward_a.mean()))
    display(pd.DataFrame(stag_results))
    pd.DataFrame(stag_results).to_csv(OUTPUT_DIR / 'stag_training.csv',index=False)
else:
    print('Optional: compare initialization effects with independent learners.')

The optional social learners observe remaining rounds and the previous joint action from
their own perspective. Both choose before the new joint action is submitted. When both
learn, each faces a changing opponent; convergence and cooperation are not guaranteed.
The adapter scores exactly the requested number of rounds and ignores the upstream
version's additional empty completion call. Source and payoffs remain unchanged.

## 6 · Keep the evidence and reflect

Results go to a new directory under `results/`, which Git ignores. The run manifest labels
fresh training versus cached comparisons. Figures and GIFs can be reused in your notes.
No Google account, Drive mount, external dataset, or GPU is required for the local lab.

In [ ]:
np.savez_compressed(OUTPUT_DIR / 'cafe_q_table.npz',q=cafe_agent.q,visits=cafe_agent.visits)
cafe_history.to_csv(OUTPUT_DIR / 'cafe_history.csv',index=False)
cafe_scoreboard.to_csv(OUTPUT_DIR / 'cafe_scoreboard.csv')
highway_curves.to_csv(OUTPUT_DIR / 'highway_learning_curves.csv',index=False)
highway_scores.to_csv(OUTPUT_DIR / 'highway_heldout_evaluation.csv',index=False)
tournament.to_csv(OUTPUT_DIR / 'prisoners_dilemma_tournament.csv',index=False)
if highway_agents:
    current_dqn = highway_agents[('DQN',budget['seeds'][0])]
    np.savez_compressed(OUTPUT_DIR / 'current_dqn_weights.npz',
        **{k:v.detach().cpu().numpy() for k,v in current_dqn.online.state_dict().items()})
manifest = {'run_mode':RUN_MODE,'versions':VERSIONS,'budget':budget,
            'cafe_source':'trained in this run','highway_comparison_source':highway_source,
            'highway_config':HIGHWAY_CONFIG,'reference_rollout':'supplied 20,000-step checkpoint, seed 0',
            'social_training':RUN_SOCIAL_TRAINING}
(OUTPUT_DIR / 'run_metadata.json').write_text(json.dumps(manifest,indent=2))
print('Saved results:',OUTPUT_DIR)

**Exit questions:**
1. What makes a state useful for predicting future reward?
2. What changes between a Q-table and a Q-network? What stays the same?
3. Why is an evaluation episode different from a training episode?
4. What changes when another agent also makes decisions?

**References:** [Gymnasium environment API](https://gymnasium.farama.org/api/env/),
[time limits](https://gymnasium.farama.org/tutorials/gymnasium_basics/handling_time_limits/),
[HighwayEnv](https://highway-env.farama.org/),
[PyTorch DQN tutorial](https://docs.pytorch.org/tutorials/intermediate/reinforcement_q_learning.html),
[PettingZoo-Dilemmas source](https://github.com/mrochk/pettingzoo-dilemmas).

Lecture connections from the supplied notebook: `3.1_RLmodels-part1`, pp. 30–35 and 46–53;
`3.2_DeepRL-part1`, pp. 15–27; `3.2_DeepRL-part2`, pp. 32–39.
The lecture PDFs are not redistributed here. See `vendor/PROVENANCE.json` for the pinned game commit.